# Canonical attacks evaluated on FiLo

This notebook mirrors the complete existing Kaggle evaluation workflow for the official [FiLo](https://github.com/CASIA-LMC-Lab/FiLo) implementation. It clones pinned source, downloads the released FiLo and fine-tuned Grounding DINO checkpoints linked by that repository, reads fixed attacks and evaluation IDs from the attached canonical Kaggle dataset, and packages numerical and qualitative results. Enable a GPU and Internet before running all cells.

The released code defaults to OpenAI `ViT-L-14-336` (`ViT-L/14@336px`). The zero-shot mapping is deliberate: each target uses both FiLo and Grounding DINO weights trained on the opposite dataset.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE REPOSITORIES AND INSTALL DEPENDENCIES =====')
WORKING = Path('/kaggle/working')
EXPERIMENT_ROOT = WORKING / 'adversarial-robustness'
FILO_ROOT = WORKING / 'FiLo'
EXPERIMENT_REPO_URL = 'https://github.com/Parsagh05/adversarial-robustness.git'
FILO_REPO_URL = 'https://github.com/CASIA-LMC-Lab/FiLo.git'
FILO_COMMIT = '36ff29ca09ba8ba3af24d7654582aea856031400'

def clone_or_update(url, destination, commit=None):
    if destination.exists():
        subprocess.run(['git', '-C', str(destination), 'fetch', '--all', '--tags'], check=True)
    else:
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    if commit:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    else:
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)

clone_or_update(EXPERIMENT_REPO_URL, EXPERIMENT_ROOT)
clone_or_update(FILO_REPO_URL, FILO_ROOT, FILO_COMMIT)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'requirements.txt')
], check=True)
# Keep Kaggle's installed PyTorch; install only FiLo's runtime dependencies.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'timm==0.9.16', 'transformers>=4.41,<5', 'addict==2.4.0',
    'yapf==0.40.2', 'prefetch_generator==1.0.3', 'huggingface-hub>=0.22',
    'ninja>=1.11',
], check=True)
# The editable install uses legacy `setup.py develop`, which fails on Kaggle's
# Python 3.12. Build the extension in the pinned official source tree instead.
GROUNDING_DINO_ROOT = FILO_ROOT / 'models' / 'GroundingDINO'
build_env = os.environ.copy()
build_env.setdefault('MAX_JOBS', '2')
subprocess.run(
    [sys.executable, 'setup.py', 'build_ext', '--inplace'],
    cwd=GROUNDING_DINO_ROOT, env=build_env, check=True,
)
if not list((GROUNDING_DINO_ROOT / 'groundingdino').glob('_C*.so')):
    raise RuntimeError('Grounding DINO CUDA extension was not built. Enable a Kaggle GPU and restart the session.')
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))
print('Experiment code:', EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline')
print('Official FiLo:', FILO_ROOT)


In [ ]:
from blackbox_evaluation_pipeline.universal_eval.artifacts import load_manifest

print('===== STEP 2: RESOLVE THE ATTACHED CANONICAL ATTACK DATASET =====')
ATTACK_SCOPES = ('per_dataset',)  # per_dataset, per_category, per_image
ATTACK_SOURCE_DATASETS = None
ATTACK_TARGET_DATASETS = ('mvtec', 'visa')
ATTACK_CATEGORIES = None
ATTACK_DIRECTIONS = None
ATTACK_LOSS_MODES = None

def valid_artifact_root(path):
    return path.is_dir() and all(
        (path / f'canonical_clip_{scope}' / 'attack_manifest.csv').is_file()
        for scope in ATTACK_SCOPES
    )

candidates = [
    Path('/kaggle/input/datasets/alirezasalehy/adversarial-attacks-vlm-survey'),
    Path('/kaggle/input/adversarial-attacks-vlm-survey'),
]
if Path('/kaggle/input').is_dir():
    candidates.extend(path.parent for path in Path('/kaggle/input').rglob('canonical_clip_per_dataset'))
ARTIFACTS_ROOT = next((path for path in candidates if valid_artifact_root(path)), None)
if ARTIFACTS_ROOT is None:
    raise FileNotFoundError('Attach alirezasalehy/adversarial-attacks-vlm-survey.')
artifacts = load_manifest(
    ARTIFACTS_ROOT, scopes=ATTACK_SCOPES, sources=ATTACK_SOURCE_DATASETS,
    targets=ATTACK_TARGET_DATASETS, categories=ATTACK_CATEGORIES,
    directions=ATTACK_DIRECTIONS, loss_modes=ATTACK_LOSS_MODES,
)
print('Canonical root:', ARTIFACTS_ROOT)
print('Available conditions:', len(artifacts))
for source, target in sorted({(a.record['source_dataset'], a.record['target_dataset']) for a in artifacts}):
    count = sum(a.record['source_dataset'] == source and a.record['target_dataset'] == target for a in artifacts)
    print(f'  {source} -> {target}: {count}')


In [ ]:
import torch
from huggingface_hub import hf_hub_download

print('===== STEP 3: RESOLVE DATASETS AND ZERO-SHOT FILO CHECKPOINTS =====')

def first_existing_directory(paths, label):
    for path in paths:
        if path.is_dir():
            return path
    raise FileNotFoundError(f'{label} was not found. Checked: {paths}')

MVTEC_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection'),
    Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection'),
], 'MVTec AD')
VISA_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922'),
    Path('/kaggle/input/visa-ad/VisA_20220922'),
], 'VisA')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')

FILO_CHECKPOINT_REPO = 'FantasticGNU/FiLo'
FILO_CHECKPOINT_REVISION = '17bbc781de7c206fa0bb94c616ed895ca7bcc913'
FILO_CHECKPOINT_CACHE = WORKING / 'filo_checkpoints'
def released_checkpoint(filename):
    return Path(hf_hub_download(
        repo_id=FILO_CHECKPOINT_REPO, filename=filename,
        revision=FILO_CHECKPOINT_REVISION,
        local_dir=FILO_CHECKPOINT_CACHE,
    ))

FILO_TRAIN_MVTEC = released_checkpoint('filo_train_on_mvtec.pth')
FILO_TRAIN_VISA = released_checkpoint('filo_train_on_visa.pth')
GROUNDING_TRAIN_MVTEC = released_checkpoint('grounding_train_on_mvtec.pth')
GROUNDING_TRAIN_VISA = released_checkpoint('grounding_train_on_visa.pth')

MODEL_KWARGS_BY_TARGET = {
    'mvtec': {
        'repository_root': str(FILO_ROOT),
        'checkpoint_path': str(FILO_TRAIN_VISA),
        'grounding_checkpoint_path': str(GROUNDING_TRAIN_VISA),
        'target_dataset': 'mvtec',
    },
    'visa': {
        'repository_root': str(FILO_ROOT),
        'checkpoint_path': str(FILO_TRAIN_MVTEC),
        'grounding_checkpoint_path': str(GROUNDING_TRAIN_MVTEC),
        'target_dataset': 'visa',
    },
}
print('MVTec:', MVTEC_ROOT)
print('VisA:', VISA_ROOT)
print('MVTec target <- VisA weights:', FILO_TRAIN_VISA, GROUNDING_TRAIN_VISA)
print('VisA target <- MVTec weights:', FILO_TRAIN_MVTEC, GROUNDING_TRAIN_MVTEC)


In [ ]:
import json

from blackbox_evaluation_pipeline import EvaluationConfig, run_evaluation

print('===== STEP 4: RUN FIXED-ID CLEAN/ADVERSARIAL EVALUATION =====')
FULL_RUN = True
OUTPUT_ROOT = WORKING / ('kaggle_new_filo_full' if FULL_RUN else 'kaggle_new_filo_check')
SAMPLES_ROOT = WORKING / ('kaggle_new_filo_samples_full' if FULL_RUN else 'kaggle_new_filo_samples_check')

def normalized_name(value):
    return ''.join(character for character in value.lower() if character.isalnum())

def find_f1_threshold(dataset, model_name):
    committed = (
        EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'thresholds'
        / normalized_name(model_name) / dataset / 'category_thresholds.json'
    )
    candidates = [committed] if committed.is_file() else list(Path('/kaggle/input').rglob('category_thresholds.json'))
    valid = []
    for path in candidates:
        try:
            payload = json.loads(path.read_text(encoding='utf-8'))
        except (OSError, ValueError):
            continue
        if (
            payload.get('dataset') == dataset
            and normalized_name(str(payload.get('target_model', ''))) == normalized_name(model_name)
            and payload.get('threshold_mode') == 'clean_f1_optimal'
        ):
            valid.append(path)
    if len(valid) != 1:
        raise RuntimeError(
            f'Expected one clean F1-optimal {model_name}/{dataset} artifact, found {valid}. '
            'Run kaggle_new_thresholds.ipynb and attach or commit its verified output.'
        )
    return str(valid[0])

THRESHOLDS_BY_TARGET = {
    dataset: find_f1_threshold(dataset, 'filo')
    for dataset in ATTACK_TARGET_DATASETS
}
print('Frozen clean F1-optimal thresholds:', THRESHOLDS_BY_TARGET)
config = EvaluationConfig(
    artifacts_root=str(ARTIFACTS_ROOT), mvtec_root=str(MVTEC_ROOT),
    visa_root=str(VISA_ROOT), output_root=str(OUTPUT_ROOT),
    model_name='filo', model_kwargs_by_target=MODEL_KWARGS_BY_TARGET,
    thresholds_by_target=THRESHOLDS_BY_TARGET, device='cuda', batch_size=1,
    metric_size=518,
    # The adapter applies FiLo's official 3x3 sigma-4 blur before DINO masking.
    anomaly_map_sigma=0.0, aupro_fpr_limit=0.30, aupro_max_thresholds=200,
    verify_checksums=True, save_predictions=True, prediction_map_size=37,
    save_qualitative_samples=True, qualitative_output_root=str(SAMPLES_ROOT),
    attack_scopes=ATTACK_SCOPES, source_datasets=ATTACK_SOURCE_DATASETS,
    target_datasets=ATTACK_TARGET_DATASETS, attack_categories=ATTACK_CATEGORIES,
    attack_directions=ATTACK_DIRECTIONS, attack_loss_modes=ATTACK_LOSS_MODES,
    max_conditions=None if FULL_RUN else 1,
    run_notes=(
        'Attached canonical CSV attack bundles; fixed evaluation IDs; official FiLo '
        'OpenAI ViT-L/14@336px plus opposite-dataset FiLo and Grounding DINO weights.'
    ),
)
SUMMARY_PATH = run_evaluation(config)
print('Finished:', SUMMARY_PATH)


In [ ]:
import csv

print('===== STEP 5: PREVIEW SUMMARY =====')
with SUMMARY_PATH.open(newline='', encoding='utf-8') as handle:
    summary_rows = list(csv.DictReader(handle))
columns = [
    'source_dataset', 'target_dataset', 'scope', 'category', 'direction', 'loss_mode',
    'clean_i_auroc', 'adversarial_i_auroc', 'delta_i_auroc',
    'clean_p_auroc', 'adversarial_p_auroc', 'delta_p_auroc',
    'clean_aupro', 'adversarial_aupro', 'delta_aupro',
    'clean_accuracy', 'adversarial_accuracy',
    'clean_fpr', 'adversarial_fpr', 'clean_fnr', 'adversarial_fnr',
    'attack_flip_rate', 'targeted_attack_success_rate',
]
for row in summary_rows:
    print({column: row[column] for column in columns})


In [ ]:
print('===== STEP 6: PACKAGE OUTPUTS =====')
results_archive = shutil.make_archive(
    str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name
)
samples_archive = shutil.make_archive(
    str(SAMPLES_ROOT), 'zip', root_dir=SAMPLES_ROOT.parent, base_dir=SAMPLES_ROOT.name
)
print('Packaged numerical results:', results_archive)
print('Packaged qualitative samples:', samples_archive)
